In [0]:
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

# Databricks widgets for execution parameters
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental", "backfill"])
dbutils.widgets.text("batch_id", "", "Batch ID (Optional)")

LOAD_TYPE = dbutils.widgets.get("load_type")
BATCH_PARAM = dbutils.widgets.get("batch_id").strip()

CATALOG = "workspace"
DB = f"{CATALOG}.complaints"
BRONZE = f"{DB}.bronze_complaints"
LOGS = f"{DB}.pipeline_execution_logs"

# Target file path based on load type
SAMPLES_DIR = "/Workspace/Users/furyboy4592@gmail.com/Consumer-Financial-Complaint-Analytics/data/samples"
TARGET_PATH = f"{SAMPLES_DIR}/incremental_load_sample.csv" if LOAD_TYPE == "incremental" else f"{SAMPLES_DIR}/full_load_sample.csv"

BATCH_ID = BATCH_PARAM if BATCH_PARAM else f"batch_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"

# Strict raw schema definition (no schema inference)
RAW_COLS = ["product", "date_sent_to_company", "issue", "sub_product", "zip_code", "tags",
            "complaint_id", "timely", "company_response", "submitted_via", "company",
            "date_received", "state", "company_public_response", "sub_issue"]
RAW_SCHEMA = StructType([StructField(c, StringType(), True) for c in RAW_COLS])

# Read CSV with schema enforcement
df_raw = spark.read.option("header", "true").schema(RAW_SCHEMA).csv(TARGET_PATH)
rows_cnt = df_raw.count()

# Add metadata columns using _metadata.file_path for Unity Catalog compatibility
df_bronze = df_raw \
    .withColumn("load_timestamp", F.current_timestamp()) \
    .withColumn("source_file", F.col("_metadata.file_path")) \
    .withColumn("batch_id", F.lit(BATCH_ID)) \
    .withColumn("load_type", F.lit(LOAD_TYPE)) \
    .withColumn("run_date", F.current_date())

# Append records into Bronze Delta table
df_bronze.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(BRONZE)

# Write execution log entry
spark.createDataFrame([(
    str(uuid.uuid4()), BATCH_ID, "Raw-to-Bronze", LOAD_TYPE, TARGET_PATH,
    datetime.now(timezone.utc), datetime.now(timezone.utc), 0.0, "Success",
    int(rows_cnt), int(rows_cnt), 0, 0, None
)], schema="""run_id string, batch_id string, layer string, load_type string, parameter string, 
             start_time timestamp, end_time timestamp, duration_seconds double, status string, 
             rows_read long, rows_inserted long, rows_updated long, rows_quarantined long, error_message string""") \
.write.format("delta").mode("append").saveAsTable(LOGS)

print(f"Bronze ingestion complete: {rows_cnt} rows inserted from {TARGET_PATH}")